In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

In [2]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
# 전처리
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'], inplace=True)

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

train: 534, val: 178, test: 179


In [6]:
results = {}

In [7]:
# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_val_pred = rf.predict(X_val)
rf_test_pred = rf.predict(X_test)

results['RandomForest'] = {
    'val_acc': accuracy_score(y_val, rf_val_pred),
    'val_f1': f1_score(y_val, rf_val_pred),
    'test_acc': accuracy_score(y_test, rf_test_pred),
    'test_f1': f1_score(y_test, rf_test_pred),
}

In [8]:
# XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

xgb_val_pred = xgb.predict(X_val)
xgb_test_pred = xgb.predict(X_test)

results['XGBoost'] = {
    'val_acc': accuracy_score(y_val, xgb_val_pred),
    'val_f1': f1_score(y_val, xgb_val_pred),
    'test_acc': accuracy_score(y_test, xgb_test_pred),
    'test_f1': f1_score(y_test, xgb_test_pred),
}

In [9]:
# LightGBM 기본
lgbm = LGBMClassifier(random_state=42, verbosity=-1)
lgbm.fit(X_train, y_train)

lgbm_val_pred = lgbm.predict(X_val)
lgbm_test_pred = lgbm.predict(X_test)

results['LightGBM'] = {
    'val_acc': accuracy_score(y_val, lgbm_val_pred),
    'val_f1': f1_score(y_val, lgbm_val_pred),
    'test_acc': accuracy_score(y_test, lgbm_test_pred),
    'test_f1': f1_score(y_test, lgbm_test_pred),
}

In [10]:
# LightGBM 하이퍼파라미터 튜닝 (num_leaves, learning_rate)
param_grid = {
    'num_leaves': [15, 31, 50],
    'learning_rate': [0.01, 0.1, 0.3],
}

best_val_f1 = 0
best_params = {}

for nl in param_grid['num_leaves']:
    for lr in param_grid['learning_rate']:
        m = LGBMClassifier(num_leaves=nl, learning_rate=lr, random_state=42, verbosity=-1)
        m.fit(X_train, y_train)
        pred = m.predict(X_val)
        f1 = f1_score(y_val, pred)
        print(f"num_leaves={nl}, lr={lr} -> val_f1={f1:.4f}")
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_params = {'num_leaves': nl, 'learning_rate': lr}

print(f"\nbest: {best_params}")

num_leaves=15, lr=0.01 -> val_f1=0.7179
num_leaves=15, lr=0.1 -> val_f1=0.7869
num_leaves=15, lr=0.3 -> val_f1=0.7385
num_leaves=31, lr=0.01 -> val_f1=0.7179
num_leaves=31, lr=0.1 -> val_f1=0.7846
num_leaves=31, lr=0.3 -> val_f1=0.7692
num_leaves=50, lr=0.01 -> val_f1=0.7179
num_leaves=50, lr=0.1 -> val_f1=0.7846
num_leaves=50, lr=0.3 -> val_f1=0.7692

best: {'num_leaves': 15, 'learning_rate': 0.1}


In [11]:
# 튜닝된 LightGBM (early stopping 없이)
lgbm_tuned = LGBMClassifier(**best_params, random_state=42, verbosity=-1)
lgbm_tuned.fit(X_train, y_train)

lgbm_tuned_val_pred = lgbm_tuned.predict(X_val)
lgbm_tuned_test_pred = lgbm_tuned.predict(X_test)

results['LGBM_tuned'] = {
    'val_acc': accuracy_score(y_val, lgbm_tuned_val_pred),
    'val_f1': f1_score(y_val, lgbm_tuned_val_pred),
    'test_acc': accuracy_score(y_test, lgbm_tuned_test_pred),
    'test_f1': f1_score(y_test, lgbm_tuned_test_pred),
}

In [12]:
# LightGBM + Early Stopping
lgbm_es = LGBMClassifier(**best_params, n_estimators=500, random_state=42, verbosity=-1)
lgbm_es.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(20), log_evaluation(0)]
)
print(f"best iteration: {lgbm_es.best_iteration_}")

lgbm_es_val_pred = lgbm_es.predict(X_val)
lgbm_es_test_pred = lgbm_es.predict(X_test)

results['LGBM_tuned_ES'] = {
    'val_acc': accuracy_score(y_val, lgbm_es_val_pred),
    'val_f1': f1_score(y_val, lgbm_es_val_pred),
    'test_acc': accuracy_score(y_test, lgbm_es_test_pred),
    'test_f1': f1_score(y_test, lgbm_es_test_pred),
}

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[38]	valid_0's binary_logloss: 0.405741
best iteration: 38


In [13]:
result_df = pd.DataFrame(results).T
result_df = result_df.round(4)
result_df

,val_acc,val_f1,test_acc,test_f1
RandomForest,0.8258,0.7559,0.8156,0.7402
XGBoost,0.8146,0.7402,0.7933,0.7087
LightGBM,0.8427,0.7846,0.8045,0.7200
LGBM_tuned,0.8539,0.7869,0.8212,0.7460
LGBM_tuned_ES,0.8596,0.7967,0.8156,0.7273
